# Building a Convolutional Neural Network to classify images from `CIFAR10`

In [85]:
import torch
from torch import nn

from torchvision import datasets
from torchvision import transforms

from torch.utils.data import DataLoader

import matplotlib.pyplot as plt

from torchmetrics.classification import Accuracy

In [86]:
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [87]:
train_data = datasets.CIFAR10(root="data", train=True, transform=transform, download=True)
test_data = datasets.CIFAR10(root="data", train=False, transform=test_transform, download=True)

train_dataloader = DataLoader(dataset=train_data, batch_size=32, shuffle=True)
test_dataloader = DataLoader(dataset=test_data, batch_size=32, shuffle=False)

In [88]:
class_names = train_data.classes
class_names

['airplane',
 'automobile',
 'bird',
 'cat',
 'deer',
 'dog',
 'frog',
 'horse',
 'ship',
 'truck']

In [89]:
image, label = train_data[0]
image.shape, label

(torch.Size([3, 32, 32]), 6)

In [90]:
class CIFAR10model(nn.Module):
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
        super().__init__()

        self.conv_layer_1 = nn.Sequential(
            nn.Conv2d(in_channels=input_shape, out_channels=hidden_units, kernel_size=(3, 3), padding=1, stride=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=(3,3), padding=1, stride=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.conv_layer_2 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=(3, 3), padding=1, stride=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=(3,3), padding=1, stride=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.classification_layer = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=hidden_units*8*8, out_features=output_shape)
        )

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        X = self.conv_layer_1(X)
        X = self.conv_layer_2(X)
        X = self.classification_layer(X)

        return X

In [91]:
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
torch.cuda.manual_seed(42)
model = CIFAR10model(input_shape=3, hidden_units=32, output_shape=len(class_names)).to(device)


In [92]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)
accuracy = Accuracy(task="multiclass", num_classes=len(class_names)).to(device)

In [93]:
epochs = 15

for epoch in range(epochs):
    train_loss, train_acc = 0, 0
    model.train()
    for X_batch, y_batch in train_dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        y_preds = model(X_batch)
        loss = loss_fn(y_preds, y_batch)
        acc = accuracy(y_preds, y_batch)

        train_loss += loss.item()
        train_acc += acc.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss /= len(train_dataloader)
    train_acc /= len(train_dataloader)

    test_loss, test_acc = 0, 0
    model.eval()
    with torch.inference_mode():
        for X_batch, y_batch in test_dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_preds = model(X_batch)
            loss = loss_fn(y_preds, y_batch)
            acc = accuracy(y_preds, y_batch)

            test_loss += loss.item()
            test_acc += acc.item()

    test_loss /= len(test_dataloader)
    test_acc /= len(test_dataloader)

    if epoch % 3 == 0:
        print(f"Epoch: {epoch}, Train loss: {train_loss:.4f}, Train accuracy: {train_acc*100:.2f}, Test loss: {test_loss:.4f}, Test accuracy: {test_acc*100:.2f}")



Epoch: 0, Train loss: 1.4011, Train accuracy: 49.62, Test loss: 1.0839, Test accuracy: 61.33
Epoch: 3, Train loss: 0.7879, Train accuracy: 72.51, Test loss: 0.8456, Test accuracy: 70.76
Epoch: 6, Train loss: 0.6699, Train accuracy: 76.85, Test loss: 0.7498, Test accuracy: 74.88
Epoch: 9, Train loss: 0.6076, Train accuracy: 78.83, Test loss: 0.7689, Test accuracy: 74.69
Epoch: 12, Train loss: 0.5656, Train accuracy: 80.53, Test loss: 0.7432, Test accuracy: 75.24


In [94]:
from pathlib import Path

model_folder = Path("models")
model_folder.mkdir(parents=True, exist_ok=True)
model_name = "mnist_model.pth"

torch.save(obj=model.state_dict(), f=model_folder/model_name)